<div style="
    border-left:8px solid #6C8E7B;
    padding-left:25px;
    margin-bottom:45px;
">
    <h1 style="
        font-size:3.3em;
        font-weight:700;
        margin-bottom:8px;
        color:#2F3E34;
    ">
        Used Car Price Analysis
    </h1>
    <p style="
        color:#7A8F82;
        font-size:1.2em;
        margin-top:0;
    ">
        Quikr Cars Dataset
    </p>
    <p style="
        color:#4F6B5A;
        font-size:1.1em;
        margin-top:20px;
    ">
        <strong>Project goals:</strong> Clean the raw Quikr used car dataset, explore relationships (price, year, mileage, brand, fuel type), and build a predictive model for listing price.
    </p>
</div>

# 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 100)


# 2. Loading the Data

In [ ]:
df = pd.read_csv('quikr_car.csv')
print(df.shape)
df.head(10)


**Result:** the dataset contains **892 rows** and **6 columns**: `name`, `company`, `year`, `Price`, `kms_driven`, `fuel_type`. Looking at the first rows, the data already looks messy - prices and mileage are stored as text with commas and units.

In [ ]:
df.info()


**Result:** all 6 columns are stored as `object` (text) - even `year`, `Price`, and `kms_driven`, which should clearly be numeric. This confirms we need to convert and clean these columns before any analysis.

In [ ]:
df.isna().sum()


**Result:** `kms_driven` has **52 missing values** and `fuel_type` has **55 missing values**. No missing values in `name`, `company`, `year`, or `Price`.


# 3. Data Cleaning

### 3.1 The `year` column

First, let's check how many rows have a non-numeric (invalid) `year` value.

In [ ]:
mask_bad_year = ~df['year'].str.isnumeric()
print('Number of invalid year values:', mask_bad_year.sum())
df.loc[mask_bad_year, 'year'].unique()


**Result:** **50 rows** have completely invalid `year` values (e.g. `'150k'`, `'TOUR'`, `'r 15'`, `'zest'`...) - these look like fragments from other columns that got shifted into `year`. We drop these rows and convert the column to integer.

In [ ]:
df = df[df['year'].str.isnumeric()].copy()
df['year'] = df['year'].astype(int)
df['year'].describe()


**Result:** after removing the invalid rows, `year` ranges from **1995 to 2019**, with a mean around 2013 - a sensible range for used car listings. Dataset now has **842 rows**.

### 3.2 The `Price` column

Next, we remove rows with `"Ask For Price"` (no actual price given), then strip the commas and convert to integer.

In [ ]:
print('Rows with "Ask For Price":', (df['Price'] == 'Ask For Price').sum())
df = df[df['Price'] != 'Ask For Price'].copy()
df['Price'] = df['Price'].str.replace(',', '', regex=False).astype(int)
df['Price'].describe()


**Result:** prices now range roughly from **30,000 Rs to ~8.5 million Rs**, with a mean around **400,000 Rs**. The huge gap between the mean and the max already hints at the presence of an extreme outlier we'll see in the EDA.

### 3.3 The `kms_driven` column

We remove the `" kms"` suffix and commas, then check for any leftover non-numeric values before converting to integer.

In [ ]:
df['kms_driven'] = df['kms_driven'].str.replace(',', '', regex=False).str.replace(' kms', '', regex=False)

mask_bad_kms = ~df['kms_driven'].str.isnumeric().fillna(False)
print('Invalid / missing kms_driven:', mask_bad_kms.sum())
df.loc[mask_bad_kms, 'kms_driven'].unique()


**Result:** **2 rows** have an invalid `kms_driven` value - the string `'Petrol'` (clearly data from the `fuel_type` column shifted into `kms_driven`). We drop these rows and convert the column to integer.

In [ ]:
df = df[~mask_bad_kms].copy()
df['kms_driven'] = df['kms_driven'].astype(int)
df['kms_driven'].describe()


**Result:** mileage ranges from very low values (some near 0, likely newer/barely-used cars) up to ~400,000 km, with a mean around 46,000 km - reasonable for a used car market.

### 3.4 The `fuel_type` column

We simply drop the rows with a missing fuel type, since this is a small fraction of the data.

In [ ]:
print('Missing fuel_type:', df['fuel_type'].isna().sum())
df = df.dropna(subset=['fuel_type']).copy()
df['fuel_type'].value_counts()


**Result:** **1 row** with a missing fuel type was dropped. The remaining distribution is **Petrol: 428**, **Diesel: 386**, **LPG: 2** - Petrol and Diesel dominate, while LPG is essentially negligible (only 2 cars).

### 3.5 The `name` column

The `name` field contains very long strings (brand + model + full trim/engine description). We shorten it to the first 3 words to get more consistent model categories.

In [ ]:
df['name'] = df['name'].str.split(' ').str.slice(0, 3).str.join(' ')
df['name'].head(10)


**Result:** names are now much shorter and more consistent, e.g. *"Hyundai Santro Xing"*, *"Maruti Suzuki Alto"* - this groups together listings that previously differed only by trim-level details.

### 3.6 Duplicates

Finally, we remove fully duplicated rows and check the overall result of the cleaning process.

In [ ]:
print('Duplicates before removal:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Number of rows after cleaning:', df.shape[0])
df.describe(include='all')


**Result:** **96 duplicate rows** were removed. The dataset shrank from the original **892 rows down to 720 clean rows** (50 rows dropped for invalid `year`, 2 for `"Ask For Price"`, 2 for invalid `kms_driven`, 1 for missing `fuel_type`, 96 duplicates - some overlap between these categories).

<div style="background-color:#D6EAF8; padding:10px; border-radius:8px;">
    <b style="color:#2F3E34;">✅ Data cleaning complete - 720 rows ready for analysis.</b>
</div>

# 4. Exploratory Data Analysis (EDA)

### 4.1 Price distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df['Price'], bins=40, kde=True)
plt.title('Distribution of Car Prices')
plt.xlabel('Price (Rs)')
plt.show()


**📊 Analysis:** the price distribution is **strongly right-skewed**. Most listings are concentrated between roughly 100,000 and 700,000 Rs, with the count dropping off sharply afterwards. A single extreme outlier (~8.5M Rs, a Mahindra XUV500) sits far to the right and visually flattens the rest of the histogram - it dominates the x-axis scale.

### 4.2 Price vs brand (company)

In [ ]:
plt.figure(figsize=(12,6))
top_companies = df['company'].value_counts().head(10).index
sns.boxplot(data=df[df['company'].isin(top_companies)], x='company', y='Price')
plt.xticks(rotation=45)
plt.title('Price vs Brand (Top 10 Most Frequent Brands)')
plt.show()


**📊 Analysis:** among the most frequent brands, **Toyota** and **Mahindra** have the highest median prices, while **Chevrolet** and **Tata** sit at the lower end. The 8.5M Rs outlier belongs to Mahindra and shows up clearly as an extreme point, stretching the entire y-axis. Budget brands (Hyundai, Maruti) have tighter, lower-priced boxes, reflecting their position as entry-level cars.

### 4.3 Price vs year of manufacture

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='year', y='Price', hue='fuel_type', alpha=0.6)
plt.title('Price vs Year of Manufacture')
plt.show()


**📊 Analysis:** there is a visible **upward trend** - newer cars (higher `year`) tend to be more expensive, especially from around 2013 onward, where the spread of prices widens considerably. Older cars (pre-2010) are tightly clustered at low prices. Diesel and Petrol points are mixed throughout, with Diesel appearing slightly more often among the higher-priced points.

### 4.4 Price vs mileage

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='kms_driven', y='Price', alpha=0.6)
plt.title('Price vs Mileage')
plt.xlabel('Mileage (km)')
plt.show()


**📊 Analysis:** the relationship here is **weak**. Most cars are clustered below 100,000 km regardless of price. Cars with very high mileage (>150,000 km) tend to be cheaper, but there is a lot of noise - mileage alone does not strongly determine price, likely because brand/model and age matter much more.

### 4.5 Correlation matrix

In [ ]:
corr = df[['year', 'Price', 'kms_driven']].corr()
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap='pink', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()


**📊 Analysis:** `year` and `Price` are **positively correlated (0.29)** - newer cars cost more, as expected, though the relationship is weak. `kms_driven` and `Price` are **negatively correlated (-0.12)** - higher mileage, lower price, but very weakly. `year` and `kms_driven` are **negatively correlated (-0.25)** - older cars tend to have higher mileage, which makes intuitive sense. None of these correlations are strong on their own.

### 4.6 Price vs fuel type

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='fuel_type', y='Price')
plt.title('Price vs Fuel Type')
plt.show()


**📊 Analysis:** **Diesel** cars have a higher median price and more high-value outliers than **Petrol** cars (including the 8.5M Rs Mahindra outlier). **LPG** cars appear cheapest, but this is based on only 2 listings, so it isn't a reliable signal.

<div style="background-color:#E8F0EB;
            border-left:5px solid #6C8E7B;
            padding:15px;
            border-radius:8px;
            margin:15px 0;">
<b style="color:#4F6B5A;">📝 EDA Summary:</b>
<ul style="color:#333333;margin:8px 0 0 0;">
<li>Prices are right-skewed with one major outlier (~8.5M Rs).</li>
<li>Newer year and lower mileage are weakly associated with higher price.</li>
<li>Brand/model and fuel type (Diesel vs Petrol) create visible price segments.</li>
<li>No single numeric feature strongly explains price on its own; a model combining all features is needed.</li>
</ul>
</div>


# 5. Price Prediction Modeling

### 5.1 Train/test split

We separate the target variable (`Price`) from the features, and split the data into 80% training and 20% testing.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)


**Result:** the data is split into a **576-row training set** and a **144-row test set** (80/20 split of 720 rows).

### 5.2 Linear Regression

We one-hot encode the categorical columns (`name`, `company`, `fuel_type`) using a `ColumnTransformer`, and combine it with a `LinearRegression` model in a `Pipeline`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

categorical = ['name', 'company', 'fuel_type']

ohe = OneHotEncoder(handle_unknown='ignore')
column_trans = ColumnTransformer(
    transformers=[('ohe', ohe, categorical)],
    remainder='passthrough'
)

model = make_pipeline(column_trans, LinearRegression())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('R2:', r2_score(y_test, y_pred))
print('MAE:', mean_absolute_error(y_test, y_pred))


**Result:** for this particular split (`random_state=42`), the model achieves **R2 ≈ 0.59** and **MAE ≈ 132,600 Rs**. At first glance this looks decent, but R2 can vary a lot depending on which rows end up in the test set - let's check that next.

### 5.3 Checking model stability across different splits

With only 720 rows and many categories in `name`, the R2 score can be very sensitive to the random train/test split. We repeat the fit/evaluation 20 times with different `random_state` values.

In [ ]:
scores = []
for r in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=r)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    scores.append(r2_score(y_test, y_pred))

import numpy as np
print('R2 mean:', np.mean(scores))
print('R2 std:', np.std(scores))
print('R2 min/max:', min(scores), max(scores))


**Result:** **R2 mean ≈ 0.39**, **std ≈ 0.18**, ranging from **0.14 to 0.66**. This confirms that the earlier R2=0.59 was a "lucky" split - on average Linear Regression explains less than 40% of the price variance, and the result is highly unstable.
<div style="background-color:#F9E79F; padding:10px; border-radius:8px;">
    <b style="color:#2F3E34;">⚠️ Likely cause:</b> the <code>name</code> column has many unique values, so the test set often contains car models the model never saw during training - with <code>handle_unknown='ignore'</code>, these rows are essentially predicted with no model-specific information, hurting accuracy.
</div>

### 5.4 Random Forest

Random Forest is generally more robust to this kind of categorical data. We repeat the same 20-split evaluation.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_rf = make_pipeline(column_trans, RandomForestRegressor(n_estimators=200, random_state=42))

scores_rf = []
for r in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=r)
    model_rf.fit(X_train, y_train)
    y_pred = model_rf.predict(X_test)
    scores_rf.append(r2_score(y_test, y_pred))

print('R2 mean:', np.mean(scores_rf))
print('R2 std:', np.std(scores_rf))


**Result:** **R2 mean ≈ 0.41**, **std ≈ 0.18** - only a small improvement over Linear Regression, and still highly unstable. Let's investigate whether the `name` column (with its many categories) is the root cause.

### 5.5 How many unique categories do we have?

In [ ]:
print('Unique name values:', df['name'].nunique())
print('Unique company values:', df['company'].nunique())
df['name'].value_counts().head(15)


**Result:** there are **254 unique `name` values** across only **720 rows** (~2.8 rows per model on average), versus just **25 unique `company` values**. With so many rare categories, one-hot encoding `name` creates hundreds of sparse columns that mostly add noise rather than signal.

### 5.6 Model without the `name` column

In [ ]:
X2 = df.drop(['Price', 'name'], axis=1)
categorical2 = ['company', 'fuel_type']

column_trans2 = ColumnTransformer(
    transformers=[('ohe', OneHotEncoder(handle_unknown='ignore'), categorical2)],
    remainder='passthrough'
)

model_rf2 = make_pipeline(column_trans2, RandomForestRegressor(n_estimators=200, random_state=42))

scores_rf2 = []
for r in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X2, y, test_size=0.2, random_state=r)
    model_rf2.fit(X_train, y_train)
    y_pred = model_rf2.predict(X_test)
    scores_rf2.append(r2_score(y_test, y_pred))

print('R2 mean:', np.mean(scores_rf2))
print('R2 std:', np.std(scores_rf2))


**Result:** **R2 mean ≈ 0.31**, **std ≈ 0.29** - removing `name` makes things **worse**, not better, and even more unstable. This tells us `name` carries genuinely useful price information (a specific model implies a specific price segment), despite its high cardinality.

### 5.7 Simplifying `name` to 2 words (brand + model, no trim)

In [ ]:
df['name_short'] = df['name'].str.split(' ').str.slice(0, 2).str.join(' ')
print('Unique name_short values:', df['name_short'].nunique())

X3 = df.drop(['Price', 'name'], axis=1)
categorical3 = ['name_short', 'company', 'fuel_type']

column_trans3 = ColumnTransformer(
    transformers=[('ohe', OneHotEncoder(handle_unknown='ignore'), categorical3)],
    remainder='passthrough'
)

model_rf3 = make_pipeline(column_trans3, RandomForestRegressor(n_estimators=200, random_state=42))

scores_rf3 = []
for r in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X3, y, test_size=0.2, random_state=r)
    model_rf3.fit(X_train, y_train)
    y_pred = model_rf3.predict(X_test)
    scores_rf3.append(r2_score(y_test, y_pred))

print('R2 mean:', np.mean(scores_rf3))
print('R2 std:', np.std(scores_rf3))


**Result:** with `name_short` (**107 unique values**), the result is **even worse**: **R2 mean ≈ 0.28**, **std ≈ 0.49**.
<div style="background-color:#E8F0EB;
            border-left:5px solid #6C8E7B;
            padding:15px;
            margin:15px 0;
            border-radius:8px;">
    <strong>Conclusion:</strong> The best configuration so far is <strong>Random Forest with the full 3-word name</strong>
    (R² mean ≈ 0.41, std ≈ 0.18, from Section 5.4). Reducing the granularity of the name consistently
    hurts performance.
</div>


### 5.8 Feature importance (best model)

We retrain the best model (Random Forest with full `name`) on the entire dataset and inspect which features matter most.

In [ ]:
model_rf.fit(X, y)  # train on the full dataset

importances = model_rf.named_steps['randomforestregressor'].feature_importances_
feature_names = model_rf.named_steps['columntransformer'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(15)
imp_df


**📊 Result - Top features:**

| Feature | Importance |
|---|---|
| `year` | 0.196 |
| `name_Mahindra XUV500 W6` | 0.162 |
| `kms_driven` | 0.109 |
| `company_Audi` | 0.084 |
| `company_Jaguar` | 0.045 |
| `name_Ford Endeavor 4x4` | 0.041 |
| `company_Mercedes` | 0.029 |
| `company_BMW` | 0.023 |
| `name_Mitsubishi Pajero Sport` | 0.022 |
| `name_Audi A3 Cabriolet` | 0.019 |

**Analysis:**
- `year` is the most important numeric feature (~20%), confirming the EDA - newer cars are priced higher.
- `kms_driven` is the second numeric feature (~11%).
- The rest of the top features are **premium brands/models** - Audi, Jaguar, BMW, Mercedes, Mitsubishi Pajero, Toyota Fortuner - the model learns that belonging to a luxury brand strongly shifts the baseline price upward.
- `Mahindra XUV500 W6` ranks unusually high (16%) - this is almost certainly driven by the **8.5M Rs outlier** we saw in the EDA, which belongs to this exact model.

<div style="
    background: linear-gradient(135deg, #6C8E7B, #5B7868);
    padding: 24px;
    border-left: 5px solid #D4AC0D;
    border-radius: 14px;
    box-shadow: 0 8px 20px rgba(0,0,0,0.12);
    margin-bottom: 40px;">
    <h2 style="
        color: #ECF0F1;
        font-family: sans-serif;
        font-weight: 600;
        font-size: 1.5em;
        margin: 0;">
        Summary and Conclusions
    </h2>
</div>



**Data cleaning:** the raw dataset (892 rows) contained numerous errors - shifted/mixed-up values in the `year` and `kms_driven` columns, "Ask For Price" entries in `Price`, and 96 duplicate rows. After cleaning, **720 rows** remained.

**EDA:**
- The price distribution is strongly right-skewed - most listings fall in the 100k-700k Rs range, with a single extreme outlier (~8.5M Rs, Mahindra XUV500).
- Price increases with the year of manufacture (correlation 0.29) and slightly decreases with mileage (correlation -0.12) - both relationships are weak and noisy.
- Diesel cars have a higher median price than Petrol; premium brands (Audi, BMW, Mercedes, Jaguar, Mini, Toyota) clearly stand out in price compared to budget brands.

**Modeling:**
- Linear Regression (one-hot encoding on `name`/`company`/`fuel_type`): R² ≈ 0.39 (std ≈ 0.18) - highly unstable depending on the train/test split.
- Random Forest with the full `name` (3 words): best result, R² ≈ 0.41 (std ≈ 0.18).
- Removing `name` or simplifying it to 2 words worsens the result (R² ≈ 0.31 and ≈ 0.28) - the specific car model carries important price information, despite having 254 unique values across only 720 rows.
- Feature importance confirms that `year`, `kms_driven`, and belonging to a premium brand/model (Audi, Jaguar, BMW, Mercedes, Mahindra XUV500, Toyota Fortuner) are the strongest predictors of price.

**Limitations:**
- The dataset is small (720 rows) and highly diverse (254 unique models), which makes generalization difficult - many models appear only 1-2 times.
- R² ≈ 0.40 means the model explains less than half of the price variance - the listed price on Quikr also depends on factors not present in the data (vehicle condition, location, listing quality, negotiation).
- The 8.5M Rs outlier (Mahindra) has a strong influence on the results and plots - future work should test the models after removing it.

**Possible next steps:**
- Remove or cap outliers and re-evaluate the models.
- Group rare `name` categories (e.g. models appearing fewer than 3 times) into an "other" bucket.
- Apply a log transformation to `Price` before regression.
- Test Gradient Boosting and XGBoost models.